In [2]:
#all imports
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../data/raw")
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

phiusiil_df = pd.read_csv(
    DATA_DIR / "PhiUSIIL_Phishing_URL_Dataset.csv"
)

uci_df = pd.read_csv(
    DATA_DIR / "uci-ml-phishing-dataset.csv"
)

print("PhiUSIIL:", phiusiil_df.shape)
print("UCI:", uci_df.shape)

PhiUSIIL: (235795, 56)
UCI: (11055, 32)


In [3]:
#standardizing the label
# PhiUSIIL: original 1 = legitimate, 0 = phishing
phiusiil_df["cyberguard_label"] = (
    phiusiil_df["label"].map({
        1: 0,  # Legitimate
        0: 1   # Phishing
    })
)

# UCI: Result 1 = legitimate, -1 = phishing
uci_df["cyberguard_label"] = (
    uci_df["Result"].map({
        1: 0,   # Legitimate
        -1: 1   # Phishing
    })
)

print("PhiUSIIL standardized labels:")
print(phiusiil_df["cyberguard_label"].value_counts())

print("\nUCI standardized labels:")
print(uci_df["cyberguard_label"].value_counts())

PhiUSIIL standardized labels:
cyberguard_label
0    134850
1    100945
Name: count, dtype: int64

UCI standardized labels:
cyberguard_label
0    6157
1    4898
Name: count, dtype: int64


In [4]:
#checking invlaid labels
assert phiusiil_df["cyberguard_label"].notna().all()
assert uci_df["cyberguard_label"].notna().all()

assert set(phiusiil_df["cyberguard_label"].unique()) == {0, 1}
assert set(uci_df["cyberguard_label"].unique()) == {0, 1}

print("Label validation passed.")

Label validation passed.


In [5]:
#cleaning phiUSIL
phiusiil_clean = phiusiil_df.copy()

phiusiil_clean["URL"] = (
    phiusiil_clean["URL"]
    .astype("string")
    .str.strip()
)

phiusiil_clean = phiusiil_clean.dropna(
    subset=["URL", "cyberguard_label"]
)

phiusiil_clean = phiusiil_clean[
    phiusiil_clean["URL"].str.len() > 0
].copy()

phiusiil_clean = phiusiil_clean.drop_duplicates(
    subset=["URL"]
).reset_index(drop=True)

print("Original shape:", phiusiil_df.shape)
print("Cleaned shape:", phiusiil_clean.shape)
print("Duplicate URLs:", phiusiil_clean["URL"].duplicated().sum())

Original shape: (235795, 57)
Cleaned shape: (235370, 57)
Duplicate URLs: 0


In [6]:
#cleaning UCI
uci_clean = uci_df.copy()

uci_clean = uci_clean.dropna(
    subset=["Result", "cyberguard_label"]
).reset_index(drop=True)

print("Original shape:", uci_df.shape)
print("Cleaned shape:", uci_clean.shape)

print("\nStandardized labels:")
print(uci_clean["cyberguard_label"].value_counts())

Original shape: (11055, 33)
Cleaned shape: (11055, 33)

Standardized labels:
cyberguard_label
0    6157
1    4898
Name: count, dtype: int64


In [7]:
#saving the dataset
phiusiil_output = (
    OUTPUT_DIR / "phiusiil_clean.csv"
)

uci_output = (
    OUTPUT_DIR / "uci_clean.csv"
)

phiusiil_clean.to_csv(phiusiil_output, index=False)
uci_clean.to_csv(uci_output, index=False)

print("Saved:", phiusiil_output.resolve())
print("Saved:", uci_output.resolve())

Saved: C:\Users\HP\OneDrive\Desktop\BPUT\CYBERGUARD\malicious_url\data\processed\phiusiil_clean.csv
Saved: C:\Users\HP\OneDrive\Desktop\BPUT\CYBERGUARD\malicious_url\data\processed\uci_clean.csv
